In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

X_raw, y = make_classification(n_samples=2000, n_features=10, n_informative=6, n_redundant=4, random_state=42)

feature_names = [f"Feature_{i+1}" for i in range(10)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = pd.DataFrame(y)

print(f"Dataset Shape: {X.shape}")

X.head()


Dataset Shape: (2000, 10)


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10
0,0.334496,0.750877,1.648378,0.457500,0.760537,0.154781,2.551356,0.322324,-0.839197,-0.098199
1,-2.141925,-0.202847,1.914500,0.338516,-2.620558,-1.214637,-3.990420,3.149442,-0.542065,2.243877
2,-1.190483,-3.517079,-1.495101,-2.202181,0.997510,-1.708252,-2.562267,0.494147,0.713598,2.631703
3,1.025684,1.405381,1.782680,0.459276,1.641769,-0.509851,2.584977,-0.970320,-1.402303,0.528150
4,-1.632327,1.908327,2.933245,-1.737488,-0.401142,0.081108,-3.825437,1.030134,-1.307362,2.747305


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

print(f"Shape of X_train = {X_train.shape}, Shape of y_train = {y_train.shape}")
print(f"Shape of X_test = {X_test.shape}, Shape of y_test = {y_test.shape}")

Shape of X_train = (1600, 10), Shape of y_train = (1600, 1)
Shape of X_test = (400, 10), Shape of y_test = (400, 1)


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

preprocesser = Pipeline(steps=[
    ('scale', StandardScaler()),
    ('PCA', PCA(n_components=.95, random_state=42))
])

X_train_processed = preprocesser.fit_transform(X_train)
X_test_processed = preprocesser.transform(X_test)

In [24]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

best_score = 0
best_n_clusters = 0
silhouette_scores = []
K_range = range(2,9)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_train_processed)
    score = silhouette_score(X_train_processed, labels)
    print(f"n_clusters = {k} | score = {score}")
    silhouette_scores.append(score)
    if (score > best_score):
        best_score = score
        best_n_clusters = k

print(f"Best n_clusters = {best_n_clusters} | best score = {best_score}")

n_clusters = 2 | score = 0.20513029251162082
n_clusters = 3 | score = 0.18761971981713146
n_clusters = 4 | score = 0.1902851445071185
n_clusters = 5 | score = 0.17898300153877236
n_clusters = 6 | score = 0.18630034870594167
n_clusters = 7 | score = 0.1781608580134984
n_clusters = 8 | score = 0.18759128350025595
Best n_clusters = 2 | best score = 0.20513029251162082


In [38]:
final_km = KMeans(n_clusters=best_n_clusters, init='k-means++', n_init=10, random_state=42)

final_km.fit(X_train_processed)

train_distance = final_km.transform(X_train_processed)
test_distance = final_km.transform(X_test_processed)

print(f"The shape of train distance: {train_distance.shape} | The shape of test distance: {test_distance.shape}")

train_clusters = final_km.predict(X_train_processed)
test_clusters = final_km.predict(X_test_processed)

print(f"train cluster: {train_clusters.shape} | test cluster: {test_clusters.shape}")

train_clusters_2D = train_clusters.reshape(-1,1)
test_clusters_2D = test_clusters.reshape(-1,1)

print(f"train cluster: {train_clusters_2D.shape} | test cluster: {test_clusters_2D.shape}")

The shape of train distance: (1600, 2) | The shape of test distance: (400, 2)
train cluster: (1600,) | test cluster: (400,)
train cluster: (1600, 1) | test cluster: (400, 1)
